In [2]:
import requests
import pandas as pd
import time
import os
from fp.fp import FreeProxy
from tqdm import tqdm
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

In [3]:
OUTPUT_DIR      = 'output'
URL_VACANCIES   = 'https://api.hh.ru/vacancies'
QUERIES         = [
    '\"Python\"', 
]

In [4]:
def get_vacancies(text: str, area: int = 1, pages: int = 2, per_page: int = 100):
    '''
    Возвращает (pages x per_page) строк информации по вакансиям \n

    text - описание вакансии                    \n
    area - регион (1 - Москва)                  \n
    pages - кол-во страниц                      \n
    per_page - кол-во возвращаемых вакансий     \n
    '''
    vacancies = []
    for page in range(pages):
        params = {'text': text, 'area': area, 'page': page, 'per_page': 100}
        response = requests.get(URL_VACANCIES, params=params)
        if response.status_code == 200:
            data = response.json()
            vacancies.extend(data['items'])
        time.sleep(0.25)  # Пауза, чтобы не нагружать API
    return vacancies

In [5]:
def get_proxy(country_id: str = 'RU', rand: bool = True, https: bool = False):
    ''' 
    Получение Proxy для смены промежуточного IP
    '''
    proxy = FreeProxy(country_id=[country_id], rand=rand, https=https).get()
    return proxy

In [6]:
PROXY = None
lock = threading.Lock()

def get_vacancy_card_info(vacancy_url: str, retry: int = 10):
    '''
    Запрос на информацию с карточки вакансии
    '''
    global PROXY
    vacancy_info = None
    proxies = {} if PROXY is None else {'http': PROXY}

    for i in range(1, retry + 1):
        # print(f"Попытка подключения {i}/{retry} {vacancy_url}")
        try:
            response = requests.get(vacancy_url, proxies=proxies)

            if response.status_code == 200:
                vacancy_info = response.json()
                break 
            raise ValueError("Некорректный код сервера")
        except Exception as e:
            # print(f"Смена proxy: {e}")
            with lock:
                PROXY = get_proxy()

            proxies['http'] = PROXY

    return vacancy_info


In [7]:
def task_vacancy_info(query: str, vacancy_url: str):
    '''
    Возвращает информацию вакансии
    '''
    response = get_vacancy_card_info(vacancy_url)
    
    return (query, response)

In [8]:
vacancies = {}
area = 2 # СПб

def parse_vacancies():
    vacancies_link = {}
    for query in QUERIES:
        vacancies_data = get_vacancies(query, area=area, per_page=5)
        vacancies_link[query] = [vacancy_data.get("url") for vacancy_data in vacancies_data]

    count_workers = len(QUERIES)
    with ThreadPoolExecutor(max_workers=count_workers) as executor:
        futures = []
        
        for query, links in vacancies_link.items():
            for link in links: 
                futures.append(executor.submit(task_vacancy_info, query, link))

    for future in as_completed(futures):
        query, vacancy_card_info = future.result()

        if query in vacancies:
            vacancies[query].append(vacancy_card_info)
        else:
            vacancies[query] = [vacancy_card_info]

parse_vacancies()

# Работа с отображением

In [9]:
def transform_vacancy_info(vacancy_info: dict) -> dict:
    row = {
        "Вакансия": vacancy_info.get("name"),
        "Вакансия, ссылка": vacancy_info.get("alternate_url"),
        "Локация": (vacancy_info.get("area") or {}).get("name"),
        "Зарплата, от": (vacancy_info.get("salary") or {}).get("from"),
        "Зарплата, до": (vacancy_info.get("salary") or {}).get("to"),
        "Зарплата, ео": (vacancy_info.get("salary") or {}).get("currency"),
        "Город": (vacancy_info.get("address") or {}).get("city"),
        "Опыт": (vacancy_info.get("experience") or {}).get("name"),
        "Занятость":(vacancy_info.get("employment") or {}).get("name"),
        "Работодатель": (vacancy_info.get("employer") or {}).get("name"),
        "Работодатель, ссылка": (vacancy_info.get("employer") or {}).get("alternate_url"),
        "Навыки": [x.get("name") for x in vacancy_info.get("key_skills")],
        "Формат работы": [x.get("name") for x in vacancy_info.get("work_format")],
    }
    return row


In [10]:
rows = []

for query, vacancies_list in vacancies.items():
    for vacancy_info in vacancies_list:
        row = {"Запрос": query}
        vacancy_info_t = transform_vacancy_info(vacancy_info)
        row.update(vacancy_info_t)
        rows.append(row)

df_total = pd.DataFrame(rows)
df_total.to_excel(os.path.join(OUTPUT_DIR, "total.xlsx"), index=False)
df_total

,Запрос,Вакансия,"Вакансия, ссылка",Локация,"Зарплата, от","Зарплата, до","Зарплата, ео",Город,Опыт,Занятость,Работодатель,"Работодатель, ссылка",Навыки,Формат работы
0,"""Python""",Системный аналитик / Бизнес-аналитик в ИТ,https://hh.ru/vacancy/130575014,Санкт-Петербург,80000.0,120000.0,RUR,Санкт-Петербург,От 1 года до 3 лет,Полная занятость,Лидсфлоу,https://hh.ru/employer/10533490,"[ClickUp, Atlassian Jira, Atlassian Confluence...","[На месте работодателя, Удалённо, Гибрид]"
1,"""Python""",Data Analyst / Product Analyst,https://hh.ru/vacancy/130566050,Санкт-Петербург,NaN,250000.0,RUR,Санкт-Петербург,От 3 до 6 лет,Полная занятость,Софтвайс,https://hh.ru/employer/4295296,"[SQL, Анализ данных, Tableau, Python, R]","[На месте работодателя, Удалённо, Гибрид]"
2,"""Python""",Программист-разработчик (Python / C++),https://hh.ru/vacancy/130823311,Санкт-Петербург,150000.0,NaN,RUR,Санкт-Петербург,От 1 года до 3 лет,Полная занятость,Специальное Программное Обеспечение,https://hh.ru/employer/11825348,"[C++, Python]",[На месте работодателя]
3,"""Python""",Middle QA тестировщик,https://hh.ru/vacancy/130177052,Санкт-Петербург,NaN,NaN,NaN,Санкт-Петербург,От 1 года до 3 лет,Полная занятость,БАНК РОССИЯ,https://hh.ru/employer/37261,[],[]
4,"""Python""",Разработчик BI (PIX BI),https://hh.ru/vacancy/130986190,Санкт-Петербург,NaN,NaN,NaN,NaN,От 1 года до 3 лет,Полная занятость,Leantech AI Lab,https://hh.ru/employer/2670835,[],"[Удалённо, Гибрид]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,"""Python""",ML-инженер,https://hh.ru/vacancy/131101570,Санкт-Петербург,200000.0,280000.0,RUR,Санкт-Петербург,От 3 до 6 лет,Полная занятость,Мозг-Аналитика,https://hh.ru/employer/12308701,"[Python, Git, Linux, Docker, MLflow, Нейронные...","[На месте работодателя, Удалённо, Гибрид]"
196,"""Python""",AI Engineer,https://hh.ru/vacancy/130781435,Санкт-Петербург,NaN,230000.0,RUR,NaN,От 1 года до 3 лет,Полная занятость,Инфостарт,https://hh.ru/employer/949684,"[Python, SQL, REST API, LLM, PostgreSQL, RAG, ...",[Удалённо]
197,"""Python""",Ведущий программист С++/ С#,https://hh.ru/vacancy/131023750,Санкт-Петербург,230000.0,NaN,RUR,Санкт-Петербург,Более 6 лет,Полная занятость,FITNESS HOUSE,https://hh.ru/employer/113923,"[C++, Qt, Базы данных, Многозадачность, Разраб...",[]
198,"""Python""",Программист C++/C#,https://hh.ru/vacancy/131023695,Санкт-Петербург,180000.0,NaN,RUR,Санкт-Петербург,От 3 до 6 лет,Полная занятость,FITNESS HOUSE,https://hh.ru/employer/113923,"[C++, Qt, Базы данных, Многозадачность, Разраб...","[На месте работодателя, Гибрид]"


In [11]:
rows_skills = []
for row in rows:
    for skill in row.get("Навыки"):
        row_skills = {
            "Запрос": row.get("Запрос"),
            "Навык": skill,
        }
        rows_skills.append(row_skills)
# 
df_skills = pd.DataFrame(rows_skills)
df_skills_agg = df_skills.groupby(by=["Запрос", "Навык"]).size().reset_index(name="Количество")
df_skills_sort = df_skills_agg.sort_values(by=["Запрос", "Количество"], ascending=[True, False])
df_skills_sort.to_excel(os.path.join(OUTPUT_DIR, "skills.xlsx"), index=False)
df_skills_sort

,Запрос,Навык,Количество
178,"""Python""",Python,87
196,"""Python""",SQL,37
92,"""Python""",Git,34
131,"""Python""",Linux,34
77,"""Python""",Docker,26
...,...,...,...
401,"""Python""",расчётное ядро,1
402,"""Python""",термодинамика,1
403,"""Python""","тестплан, тесткейс, баг-репорт",1
404,"""Python""",техподдержка,1
